# SecureRAG Tutorial 2: Backend Parity

This notebook compares retrieval + generation behavior across available backends:
- `rust://local` (in-process Rust bridge)
- `http://127.0.0.1:8099` (HTTP simulation server)
- `grpc://127.0.0.1:50051` (strict typed gRPC served by Rust binary)

Prerequisites:
- HTTP simulation server for `http://127.0.0.1:8099`
- Optional Rust gRPC server for `grpc://127.0.0.1:50051`

Rust gRPC startup:

```bash
source .venv/bin/activate
cargo build --manifest-path securerag-rs/Cargo.toml --bin securerag_grpc_server
./securerag-rs/target/debug/securerag_grpc_server --host 127.0.0.1 --port 50051
```


In [5]:
from securerag import PrivacyConfig, PrivacyProtocol, SecureRAGAgent
from securerag.corpus import CorpusBuilder
from securerag.llm import ModelAgentLLM
from securerag.models import RawDocument

In [6]:
docs = [
    RawDocument(doc_id="q3", text="Q3 risk report highlights vendor concentration and delayed remediation."),
    RawDocument(doc_id="policy", text="Security policy requires quarterly risk treatment tracking and owner assignment."),
    RawDocument(doc_id="ops", text="Operational incidents increased in July due to queue saturation in ingestion service."),
]

query = "Q3 risk"
backends = [
    "rust://local",
    "http://127.0.0.1:8099",
    "grpc://127.0.0.1:50051",
]

In [11]:
def run_once(backend: str):
    cfg = PrivacyConfig(
        protocol=PrivacyProtocol.BASELINE,
        backend=backend,
        top_k=3,
        max_rounds=3,
    )
    corpus = (
        CorpusBuilder(cfg.protocol, backend_url=cfg.backend)
        .with_chunk_size(120)
        .add_documents(docs)
        .build()
    )
    llm = ModelAgentLLM(provider="ollama", model="qwen3:0.6b", use_ollama=True, use_huggingface=False,timeout_s=60)
    agent = SecureRAGAgent.from_config(cfg, corpus=corpus, llm=llm)

    # Full pipeline: retrieval + generation
    result = agent.run("Summarize key risks in Q3")
    retrieved = agent.retriever.retrieve(query, round_n=0)

    return {
        "backend": backend,
        "corpus_type": type(corpus).__name__,
        "top_doc": retrieved[0].doc_id if retrieved else None,
        "top_score": round(retrieved[0].score, 6) if retrieved else None,
        "context_size": result.context_size,
        "rounds": result.rounds,
        "answer_preview": result.answer.splitlines()[0] if result.answer else "",
    }

In [12]:
results = []
for backend in backends:
    try:
        results.append(run_once(backend))
    except Exception as exc:
        results.append({"backend": backend, "error": str(exc)})

# Compare full RAG outputs per backend
results

[{'backend': 'rust://local',
  'corpus_type': 'GenericCorpus',
  'top_doc': 'q3',
  'top_score': 0.181818,
  'context_size': 3,
  'rounds': 3,
  'answer_preview': 'Key risks in Q3 include vendor concentration, delayed remediation, and operational incidents due to queue saturation in ingestion services.'},
 {'backend': 'http://127.0.0.1:8099',
  'error': 'Backend call failed for chunk: [Errno 61] Connection refused'},
 {'backend': 'grpc://127.0.0.1:50051',
  'corpus_type': 'GenericCorpus',
  'top_doc': 'q3',
  'top_score': 0.222222,
  'context_size': 3,
  'rounds': 3,
  'answer_preview': 'Key risks in Q3 include vendor concentration, delayed remediation, and operational incidents due to queue saturation. These factors highlight potential challenges in risk management and operational efficiency.'}]